In [25]:
import sqlite3
import pandas as pd

# Load your cleaned CSV into a SQLite database
conn = sqlite3.connect("supplement_sales.db")
data = pd.read_csv("C:\\Users\\ASUS\\Desktop\\Coding Projects\\Build or Die\\Week 3\\Data Science\\Supplement Sales\\Supplement_Sales_Weekly_Cleaned.csv")
data.to_sql("supplement_sales", conn, if_exists="replace", index=False)

pd.options.display.float_format = '{:.4f}'.format

# 1. What are the top 5 products by total revenue?

In [26]:
query = """
    SELECT `Product Name`, SUM(`Net Revenue`) AS total_revenue
    FROM supplement_sales
    GROUP BY `Product Name`
    ORDER BY total_revenue DESC
    LIMIT 5
"""
pd.read_sql(query, conn)

,Product Name,total_revenue
0,Biotin,1473630.1678
1,Zinc,1470052.3297
2,Pre-Workout,1463613.3316
3,BCAA,1451277.9854
4,Fish Oil,1438774.2981


# 2. What is the total revenue per platform?

In [27]:
query = """
    SELECT Platform, SUM("Net Revenue") AS total_revenue
    FROM supplement_sales
    GROUP BY Platform
    ORDER BY total_revenue DESC
"""
pd.read_sql(query, conn)

,Platform,total_revenue
0,iHerb,7784968.7835
1,Amazon,7599479.5632
2,Walmart,7325000.2505


# 3. Which location generates the most units sold?

In [42]:
query = """
    SELECT Location, SUM("Units Sold") AS total_units_sold
    FROM supplement_sales
    GROUP BY Location
    ORDER BY total_units_sold DESC
    LIMIT 1
"""
pd.read_sql(query, conn)

,Location,total_units_sold
0,Canada,226053


# 4. What is the average discount per category?

In [29]:
query = """
    SELECT Category, AVG(Discount) AS avg_discount
    FROM supplement_sales
    GROUP BY Category
    ORDER BY avg_discount DESC
"""

pd.read_sql(query, conn)

,Category,avg_discount
0,Omega,0.1289
1,Mineral,0.1266
2,Herbal,0.1262
3,Fat Burner,0.1257
4,Vitamin,0.1254
5,Sleep Aid,0.1245
6,Performance,0.1227
7,Amino Acid,0.1216
8,Protein,0.1215
9,Hydration,0.1191


# 5. Which product has the highest return rate? 

In [31]:
query = """
    SELECT "Product Name", CAST(SUM("Units Returned") AS FLOAT) / SUM("Units Sold") AS return_rate
    FROM supplement_sales
    GROUP BY "Product Name"
    ORDER BY return_rate DESC
"""

pd.read_sql(query, conn)

,Product Name,return_rate
0,Vitamin C,0.0112
1,Electrolyte Powder,0.0107
2,Collagen Peptides,0.0105
3,Magnesium,0.0104
4,BCAA,0.0104
5,Iron Supplement,0.0103
6,Multivitamin,0.0103
7,Pre-Workout,0.0103
8,Green Tea Extract,0.0102
9,Creatine,0.0101


# 6. What is the monthly total revenue trend? 

In [35]:
query = """
    SELECT Monthly, SUM("Net Revenue") AS total_revenue
    FROM supplement_sales
    GROUP BY Monthly
    ORDER BY total_revenue DESC
"""

pd.read_sql(query, conn)

,Monthly,total_revenue
0,2023-05,447351.1854
1,2024-09,446091.5324
2,2023-10,443571.5341
3,2024-12,443462.7728
4,2021-08,437845.0750
...,...,...
58,2021-04,309116.6483
59,2023-11,306465.1475
60,2025-01,302565.9468
61,2022-04,301753.4434


# 7. Which platform and location combo generates the most revenue?

In [36]:
query = """
    SELECT Platform, Location, SUM("Net Revenue") AS total_revenue
    FROM supplement_sales
    GROUP BY Platform, Location
    ORDER BY total_revenue DESC
"""

pd.read_sql(query, conn)

,Platform,Location,total_revenue
0,iHerb,Canada,2692365.8152
1,Walmart,UK,2614312.0267
2,iHerb,UK,2599185.2742
3,Amazon,Canada,2590084.8507
4,Amazon,USA,2589319.9500
5,Walmart,Canada,2497333.2116
6,iHerb,USA,2493417.6941
7,Amazon,UK,2420074.7625
8,Walmart,USA,2213355.0122


# 8. Which category has the lowest average discount but highest revenue?

In [37]:
query = """
    SELECT Category, SUM("Net Revenue") AS total_revenue, AVG(Discount) AS avg_discount
    FROM supplement_sales
    GROUP BY Category
    ORDER BY total_revenue DESC, avg_discount ASC
"""

pd.read_sql(query, conn)

,Category,total_revenue,avg_discount
0,Vitamin,4260951.2371,0.1254
1,Mineral,4238538.7753,0.1266
2,Performance,2883582.2233,0.1227
3,Protein,2830554.1246,0.1215
4,Amino Acid,1451277.9854,0.1216
5,Omega,1438774.2981,0.1289
6,Fat Burner,1427996.8773,0.1257
7,Hydration,1398560.9388,0.1191
8,Herbal,1394053.1322,0.1262
9,Sleep Aid,1385159.0051,0.1245


# 9. What are the top 3 products per category by total revenue?

In [43]:
query = """
    SELECT "Product Name", Category, total_revenue
    FROM (
        SELECT "Product Name", Category, 
            SUM("Net Revenue") AS total_revenue,
            RANK() OVER (PARTITION BY Category ORDER BY SUM("Net Revenue") DESC) AS rnk
        FROM supplement_sales
        GROUP BY "Product Name", Category
    )
    WHERE rnk <= 3
    ORDER BY Category, total_revenue DESC;
"""

pd.read_sql(query, conn)

,Product Name,Category,total_revenue
0,BCAA,Amino Acid,1451277.9854
1,Green Tea Extract,Fat Burner,1427996.8773
2,Ashwagandha,Herbal,1394053.1322
3,Electrolyte Powder,Hydration,1398560.9388
4,Zinc,Mineral,1470052.3297
5,Iron Supplement,Mineral,1418837.0189
6,Magnesium,Mineral,1349649.4267
7,Fish Oil,Omega,1438774.2981
8,Pre-Workout,Performance,1463613.3316
9,Creatine,Performance,1419968.8917


# 10. What is the net revenue per platform after accounting for returns?

In [41]:
query = """
    SELECT Platform, SUM("Net Revenue") AS total_revenue, CAST(SUM("Units Returned") AS FLOAT) / SUM("Units Sold") AS return_rate
    FROM supplement_sales
    GROUP BY Platform
    ORDER BY total_revenue DESC, return_rate
"""

pd.read_sql(query, conn)

,Platform,total_revenue,return_rate
0,iHerb,7784968.7835,0.0102
1,Amazon,7599479.5632,0.0104
2,Walmart,7325000.2505,0.0100
